In [1]:
%pip install langchain langchain-openai tqdm

  Using cached langgraph-1.1.10-py3-none-any.whl.metadata (8.0 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.0.3-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.0.13-py3-none-any.whl.metadata (5.2 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/543.9 kB ? eta -:--:--
   ---------------------------------------- 543.9/543.9 kB 4.6 MB/s eta 0:00:00
Using cached langgraph-1.1.10-py3-none-any.whl (173 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from tqdm import tqdm

# ==========================================
# 1. Cấu hình LLM với Custom Endpoint
# ==========================================
llm = ChatOpenAI(
    base_url="https://llm.phuocnguyn.id.vn/v1",
    api_key="sk-dummy", 
    model="google/gemma-3-27b-it-qat-q4_0-gguf:Q4_0",
    temperature=0.1,
    max_tokens=2048,
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json"
    }
)

# ==========================================
# 2. Xây dựng Prompt Template
# ==========================================
system_template = """Bạn là một chuyên gia dịch thuật Toán học từ Tiếng Trung sang Tiếng Việt.
Nhiệm vụ của bạn là nhận vào một đối tượng JSON chứa một bài toán và dịch các giá trị văn bản sang Tiếng Việt.

YÊU CẦU NGHIÊM NGẶT:
1. Tuyệt đối giữ nguyên các khóa (keys) của JSON (content, kc_routes, answer, analysis, type, options...).
2. BẮT BUỘC giữ nguyên mọi định dạng toán học, số liệu nằm trong cặp dấu $$...$$, $...$, hoặc các tag như \n.
3. Không dịch các từ khóa phân loại nếu chúng mang tính hệ thống (ví dụ giữ nguyên "填空" nếu bạn muốn map sau, hoặc dịch thành "Điền khuyết" tuỳ mục đích, ở đây ưu tiên dịch hết nội dung đọc hiểu).
4. CHỈ trả về dữ liệu định dạng JSON hợp lệ, không thêm bất kỳ văn bản chào hỏi hay giải thích nào.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{json_data}")
])

# Sử dụng JsonOutputParser để ép LLM trả về Dict Python
chain = prompt | llm | JsonOutputParser()

# ==========================================
# 3. Hàm xử lý và dịch Dữ liệu
# ==========================================
def translate_dataset(input_filepath, output_filepath):
    # Đọc file JSON gốc
    with open(input_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    translated_data = {}
    
    # Duyệt qua từng câu hỏi (0, 1, 2, 3...)
    print(f"Bắt đầu dịch {len(data)} bản ghi...")
    for key, item in tqdm(data.items()):
        try:
            # Chuyển object thành string để đưa vào prompt
            item_json_str = json.dumps(item, ensure_ascii=False)
            
            # Gọi LangChain
            result = chain.invoke({"json_data": item_json_str})
            
            # Lưu kết quả
            translated_data[key] = result
            
            # Ghi vào file ngay lập tức để backup (tránh crash giữa chừng)
            with open(output_filepath, 'w', encoding='utf-8') as f:
                json.dump(translated_data, f, ensure_ascii=False, indent=4)
                
            # Tuỳ chọn: Thêm sleep nhỏ nếu server của bạn có rate limit
            # time.sleep(0.5) 
            
        except Exception as e:
            print(f"\nLỗi ở bản ghi {key}: {e}")
            # Nếu lỗi, giữ nguyên bản gốc để xử lý sau
            translated_data[key] = item 

    print("\nHoàn tất quá trình dịch!")

# ==========================================
# 4. Thực thi
# ==========================================
if __name__ == "__main__":
    # Thay đổi đường dẫn tới file của bạn
    INPUT_FILE = "D:\\IntelligentTesting\\IntelligentTesting\\XES3G5M\\XES3G5M\\metadata\\translation\\questions_demo.json" 
    OUTPUT_FILE = "test_translated.json"
    
    # Tạo một file sample nhỏ để test script trước khi chạy toàn bộ
    translate_dataset(INPUT_FILE, OUTPUT_FILE)

Bắt đầu dịch 11 bản ghi...


 27%|██▋       | 3/11 [00:00<00:00, 10.68it/s]


Lỗi ở bản ghi 0: Your request was blocked.

Lỗi ở bản ghi 1: Your request was blocked.

Lỗi ở bản ghi 2: Your request was blocked.

Lỗi ở bản ghi 3: Your request was blocked.


 73%|███████▎  | 8/11 [00:00<00:00, 17.45it/s]


Lỗi ở bản ghi 4: Your request was blocked.

Lỗi ở bản ghi 5: Your request was blocked.

Lỗi ở bản ghi 6: Your request was blocked.

Lỗi ở bản ghi 7: Your request was blocked.

Lỗi ở bản ghi 8: Your request was blocked.

Lỗi ở bản ghi 9: Your request was blocked.


100%|██████████| 11/11 [00:00<00:00, 14.90it/s]


Lỗi ở bản ghi 10: Your request was blocked.

Hoàn tất quá trình dịch!
